In [ ]:
### Project for Data Analysis, ML on data about Car sales history
### Project for Data Analysis, ML on data about Car sales history
### Version 1 ; Multiple linear regression and Random forest method is used to predict car prise values 
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [ ]:
#Comprehensive pricelist of 4000 different passenger vehicles 
# ── 1. Load raw data ──────────────────────
df = pd.read_csv('used_cars.csv')

In [144]:
df.head()

,model,model_year,milage,engine,transmission,ext_col,int_col,accident,clean_title,price,...,brand_Volvo,brand_smart,fuel_type_E85 Flex Fuel,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid,fuel_type_not supported,fuel_type_–,horsepower,displacement_L
0,Utility Police Interceptor Base,2013,51000.0,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,1.0,Yes,10300.0,...,False,False,True,False,False,False,False,False,300.0,3.7
1,Palisade SEL,2021,34742.0,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,1.0,Yes,38005.0,...,False,False,False,True,False,False,False,False,310.0,3.8
2,RX 350 RX 350,2022,22372.0,3.5 Liter DOHC,Automatic,Blue,Black,0.0,NaN,54598.0,...,False,False,False,True,False,False,False,False,310.0,3.5
3,Q50 Hybrid Sport,2015,88900.0,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,0.0,Yes,15500.0,...,False,False,False,False,True,False,False,False,354.0,3.5
4,Q3 45 S line Premium Plus,2021,9835.0,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,0.0,NaN,34999.0,...,False,False,False,True,False,False,False,False,310.0,2.0


In [ ]:
# Horsepower is extracted from the engine description 
df['horsepower'] = df['engine'].str.extract(r'([\d.]+)\s*HP', expand=False).astype(float)


In [ ]:
# Cars with undefined Horspower information are assigned with a median horsepower  
df['horsepower'].fillna(df['horsepower'].median(), inplace=True)

/var/folders/dx/yf419hq51tq66mh7d_71847c0000gn/T/ipykernel_33324/1826784813.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['horsepower'].fillna(df['horsepower'].median(), inplace=True)


In [ ]:
# Engine displacement (litres) is extracted from the engine description 
df['displacement_L'] = df['engine'].str.extract(r'([\d.]+)\s*L', expand=False).astype(float)


In [ ]:
# Cars with undefined engine displacement info. are assigned with a median engine displacement 
df['displacement_L'].fillna(df['displacement_L'].median(), inplace=True)

/var/folders/dx/yf419hq51tq66mh7d_71847c0000gn/T/ipykernel_33324/3154928364.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['displacement_L'].fillna(df['displacement_L'].median(), inplace=True)


In [ ]:
# The non-numerical feature (fuel type) is converted to multiple binary  
#To avoid dummy variable trap (perfect multicollinearity), we drop one of the categories of the featture 
#Its important to use binary encoding to avoid mathematical false order between categories (i.e. Diesel > Gasoline)
df = pd.get_dummies(df, columns=['fuel_type'], drop_first=True)


In [ ]:
# The non-numerical feature (brand) is converted to multiple binary  
df = pd.get_dummies(df, columns=['brand'], drop_first=True)


KeyError: "None of [Index(['brand'], dtype='object')] are in the [columns]"

In [64]:
# Clean mileage column
df['milage'] = (df['milage']
                 .str.replace(',', '', regex=False)
                 .str.replace('mi.', '', regex=False)
                 .str.strip()
                 .astype(float))

In [65]:
## we chose 0 for no accident and 1 for accident 
df['accident'] = df['accident'].map({
    'None reported': 0,
    'At least 1 accident or damage reported': 1
})

In [ ]:
# Clean price coloumn 

# Price: "$10,300" → 10300.0
df['price'] = (df['price']
               .str.replace(r'[\$,]', '', regex=True)
               .astype(float))

In [ ]:
#here all the binary features related to fuel type are considered
fuel_cols = [col for col in df.columns if col.startswith('fuel_')]

In [ ]:
#here all the binary features related to brand type are considered

brand_cols = [col for col in df.columns if col.startswith('brand_')]


In [ ]:
#Total feature array is constructed 
features = ['model_year', 'milage','accident','horsepower','displacement_L'] + brand_cols + fuel_cols

#NaN are removed from the dataset as they cause error 
clean_df = df[features + ['price']].dropna()

X = clean_df[features].values
y = clean_df['price'].values

In [146]:
clean_df.head()

,model_year,milage,accident,horsepower,displacement_L,brand_Alfa,brand_Aston,brand_Audi,brand_BMW,brand_Bentley,...,brand_Volkswagen,brand_Volvo,brand_smart,fuel_type_E85 Flex Fuel,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid,fuel_type_not supported,fuel_type_–,price
0,2013,51000.0,1.0,300.0,3.7,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,10300.0
1,2021,34742.0,1.0,310.0,3.8,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,38005.0
2,2022,22372.0,0.0,310.0,3.5,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,54598.0
3,2015,88900.0,0.0,354.0,3.5,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,15500.0
4,2021,9835.0,0.0,310.0,2.0,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,34999.0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
#First we are going to try multiple linear regression approach for building the predictive model 
from sklearn.linear_model import LinearRegression

mlr = LinearRegression()
mlr.fit(X_train, y_train)

y_pred = mlr.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

r2_mlr   = r2_score(y_test, y_pred)
rmse_mlr = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"mlr  R²:   {r2_mlr:.4f}")
print(f"mlr RMSE: {rmse_mlr:,.0f}")


mlr  R²:   0.6104
mlr RMSE: 30,407


In [ ]:
# Second, we are trying to build the model using Random Forest approach 
from sklearn.ensemble import RandomForestRegressor

rf_regressor = RandomForestRegressor(n_estimators=500, random_state=0)
rf_regressor.fit(X_train, y_train)


RandomForestRegressor(n_estimators=500, random_state=0)

In [149]:
y_pred_rf = rf_regressor.predict(X_test)

In [150]:

r2_rf   = r2_score(y_test, y_pred_rf)

print(f"rf  R²:   {r2_rf:.4f}")


rf  R²:   0.8402


In [ ]:
#Multiple linear regression leads to Rsquare of 0.6 
# Random Forest has Rsquared of 0.85 (Winner) 

In [ ]:
# Now we want to predict the price of a Random car such as Audi etron only via its (brand, milage, model year, Hp, accident)
input_dict = {col: 0 for col in features}


In [160]:
input_dict['model_year']     = 2020
input_dict['milage']         = 40000
input_dict['accident']       = 0
input_dict['horsepower']     = 310
input_dict['displacement_L'] = df['displacement_L'].median()

In [168]:
if 'brand_Audi' in input_dict:
    input_dict['brand_Audi'] = 1

if 'fuel_type_Electric' in input_dict:
    input_dict['fuel_type_Electric'] = 1

In [169]:
input_df = pd.DataFrame([input_dict])


In [174]:
predicted_price = rf_regressor.predict(input_df.values)
print(f"Predicted Price for Audi e-tron: ${predicted_price[0]:,.0f}")

Predicted Price for Audi e-tron: $42,300
